In [1]:
#==============================================================================
# Loading the data
#==============================================================================

import pandas as pd

url = "https://raw.githubusercontent.com/Amirhossein-momeni/AmirGeek_ML_Datasets/refs/heads/main/DataSet/01-Ad.csv"
df = pd.read_csv(url)

In [2]:
#==============================================================================
# Exploring the data (EDA)
#==============================================================================

print(70* "-")  
print(df.shape) 
print(70* "-")                                     
print(df.head())
print(70* "-")  
print(df.info())
print(70* "-")  
print(df.isna().sum())
print(70* "-")    
print(df.isna().sum().sum()) 
print(70* "-")                              
print(df["Purchased"].value_counts(normalize=True))  
print(70* "-")  

----------------------------------------------------------------------
(400, 6)
----------------------------------------------------------------------
   UserID  Gender Type  Age  EstimatedSalary  Purchased
0       1    Male    B   24          2800000          0
1       2    Male    A   48          3000000          0
2       3  Female    B   34          7800000          0
3       4  Female    B   36         10700000          0
4       5    Male    D   24         14700000          0
----------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   UserID           400 non-null    int64 
 1   Gender           400 non-null    object
 2   Type             400 non-null    object
 3   Age              400 non-null    int64 
 4   EstimatedSalary  400 non-null    int64 
 5   Purchased        4

In [3]:
#==============================================================================
# Defining features and target
#==============================================================================

X = df.drop(columns=["UserID", "Purchased"])
y = df["Purchased"]

In [4]:
#==============================================================================
# Train / Test split
#==============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
#==============================================================================
# Preprocessing (two candidate feature sets)
#==============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["Age", "EstimatedSalary"]
cat_cols = ["Gender", "Type"]

prep_num_only = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
])

prep_with_cat = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols),
])

In [7]:
#==============================================================================
# Building the model + choosing the feature set with CV
#==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

model_num = Pipeline([("prep", prep_num_only), ("nb", GaussianNB())])
model_cat = Pipeline([("prep", prep_with_cat), ("nb", GaussianNB())])

for name, m in [("numeric only", model_num), ("numeric + categorical", model_cat)]:
    score = cross_val_score(m, X_train, y_train, cv=5, scoring="f1").mean()
    print(name, round(score, 3))

numeric only 0.838
numeric + categorical 0.807


In [8]:
#==============================================================================
# Experiment: what happens WITHOUT scaling?
#==============================================================================

prep_raw = ColumnTransformer([("num", "passthrough", num_cols)])
model_raw = Pipeline([("prep", prep_raw), ("nb", GaussianNB())])

model_raw.fit(X_train, y_train)
y_pred_raw = model_raw.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix
print(accuracy_score(y_test, y_pred_raw))      # 0.75
print(confusion_matrix(y_test, y_pred_raw))    # [[49  2] [18 11]]

0.75
[[49  2]
 [18 11]]


In [9]:
#==============================================================================
# Training + inspecting what the model learned
#==============================================================================

model = model_num
model.fit(X_train, y_train)

nb = model.named_steps["nb"]
print(nb.class_prior_)   # [0.64375 0.35625] -> the prior, straight from the data
print(nb.theta_)         # per-class means (in scaled units)
print(nb.var_)           # per-class variances (in scaled units)

[0.64375 0.35625]
[[-0.4606825  -0.30402043]
 [ 0.83246136  0.54937026]]
[[0.57244608 0.50349369]
 [0.69610514 1.42836814]]


In [10]:
#==============================================================================
# Evaluation (baseline model)
#==============================================================================

from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(accuracy_score(y_test, y_pred))          # 0.875
print(confusion_matrix(y_test, y_pred))        # [[47  4] [ 6 23]]
print(classification_report(y_test, y_pred))

0.875
[[47  4]
 [ 6 23]]
              precision    recall  f1-score   support

           0       0.89      0.92      0.90        51
           1       0.85      0.79      0.82        29

    accuracy                           0.88        80
   macro avg       0.87      0.86      0.86        80
weighted avg       0.87      0.88      0.87        80



In [11]:
#==============================================================================
# Hyperparameter tuning (var_smoothing)
#==============================================================================

from sklearn.model_selection import GridSearchCV

param_grid = {"nb__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]}

search = GridSearchCV(model_num, param_grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

print(search.best_params_)   # {'nb__var_smoothing': 1e-09}  (the default!)
print(search.best_score_)    # ~0.838

{'nb__var_smoothing': 1e-09}
0.8382677708764665


In [12]:
#==============================================================================
# Final evaluation (once!) + probabilities
#==============================================================================

best_model = search.best_estimator_
y_pred_final = best_model.predict(X_test)

print(accuracy_score(y_test, y_pred_final))      # 0.875
print(confusion_matrix(y_test, y_pred_final))    # [[47  4] [ 6 23]]

proba = best_model.predict_proba(X_test)[:, 1]   # P(Purchased = 1)
print(proba[:3].round(3))                        # [0.962 0.031 0.116]
print(y_test.iloc[:3].tolist())                  # [1, 0, 0]

0.875
[[47  4]
 [ 6 23]]
[0.962 0.031 0.116]
[1, 0, 0]
